## Datos de Ingresos de Adultos (Adult Census Income)

El conjunto de datos Adult (también conocido como "Census Income") contiene información demográfica y laboral extraída del censo de EE.UU. de 1994. El objetivo es predecir si el ingreso anual de una persona supera los 50,000 dólares.

**Dataset:** [Adult Census Income - UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/adult)

---

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Carga y Selección de Datos

Se carga el archivo `.data` sin encabezado y se asignan los nombres de columnas manualmente. Los valores faltantes en este dataset están representados como `' ?'` (espacio + signo de interrogación).

| Columna (índice) | Nombre             | Descripción |
|------------------|--------------------|-------------|
| 0                | `age`              | Edad del individuo |
| 1                | `workclass`        | Clase de trabajo (Private, Self-emp, Gov, etc.) |
| 2                | <span style="color:red">`fnlwgt`</span> | Peso final del censo (variable de muestreo, no relevante para predicción) |
| 3                | `education`        | Nivel educativo más alto alcanzado |
| 4                | `education_num`    | Número de años de educación |
| 5                | `marital_status`   | Estado civil |
| 6                | `occupation`       | Tipo de ocupación |
| 7                | `relationship`     | Relación en el hogar (Husband, Wife, etc.) |
| 8                | `race`             | Raza |
| 9                | `sex`              | Género |
| 10               | `capital_gain`     | Ganancia de capital |
| 11               | `capital_loss`     | Pérdida de capital |
| 12               | `hours_per_week`   | Horas trabajadas por semana |
| 13               | `native_country`   | País de origen |
| 14               | `income`           | **Variable objetivo:** `<=50K` o `>50K` |

**Variable objetivo (col. 14):** `income` (nivel de ingresos: <=50K o >50K)

In [11]:
import pandas as pd

# Nombres de columnas (el archivo no tiene encabezado)
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

dt = pd.read_csv('../Database/PP5_adult.data',
    header=None,
    names=column_names,
    na_values=' ?',       # los faltantes vienen como ' ?'
    skipinitialspace=True # elimina espacios al inicio de cada campo
)

# Limpiar espacios en columnas de texto
dt.columns = dt.columns.str.strip()

# Eliminar columna fnlwgt (peso de muestreo, no es predictora)
columnas_a_excluir = ['fnlwgt']
dt = dt.drop(columns=columnas_a_excluir)

print(dt.columns.tolist())
# print(pd.isnull(dt).sum())

['age', 'workclass', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [12]:
numeric_cols = dt.select_dtypes(include=['number']).columns.tolist()
categorical_cols = dt.select_dtypes(include=['object']).columns.tolist()
categorical_cols = dt.select_dtypes(include=['object']).columns.tolist()

In [13]:
# Imputación de columnas numéricas con la mediana
for col in numeric_cols:
    n_faltantes = dt[col].isna().sum()
    if n_faltantes > 0:
        mediana = dt[col].median()
        dt[col].fillna(mediana, inplace=True)
        print(f"  [{col}] → {n_faltantes} faltantes imputados con mediana={mediana}")

# Escalar columnas numéricas para KNN
scaler = StandardScaler()
X_num = scaler.fit_transform(dt[numeric_cols])

# KNN sobre las columnas numéricas (k=5 vecinos)
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')  # 6 porque incluye la propia fila
knn.fit(X_num)
distancias, indices = knn.kneighbors(X_num)  # precalcular vecinos para todas las filas

print(f"Columnas categóricas a procesar: {len(categorical_cols)}")
print(categorical_cols)

# Imputar cada columna categórica con KNN-moda
for col in categorical_cols:
    mask_na = (dt[col] == '?') | (dt[col] == ' ?') | (dt[col].isna())
    n_faltantes = mask_na.sum()

    if n_faltantes == 0:
        pass  # sin faltantes, saltar directo a encodear
    else:
        print(f"  [{col}] → {n_faltantes} faltantes, imputando con KNN-moda...")
        filas_na = dt.index[mask_na].tolist()

        for fila in filas_na:
            # Los 6 vecinos incluyen la fila misma en índice 0 → usamos [1:6]
            vecinos_idx = indices[fila][1:6]
            # Tomar los valores de esos vecinos en esta columna (excluir NA)
            valores_vecinos = dt[col].iloc[vecinos_idx]
            valores_vecinos = valores_vecinos[
                ~((valores_vecinos == '?') | (valores_vecinos == ' ?') | valores_vecinos.isna())
            ]

            if len(valores_vecinos) > 0:
                moda = valores_vecinos.mode()[0]  # valor más frecuente entre vecinos
            else:
                # fallback: moda global de la columna (excluyendo NA)
                valores_globales = dt[col][~mask_na]
                moda = valores_globales.mode()[0]

            dt.at[fila, col] = moda

    # Convertir la columna a entero con LabelEncoder
    le = LabelEncoder()
    dt[col] = le.fit_transform(dt[col].astype(str))

print("\n✓ Imputación y codificación completada.")
print(f"¿Quedan '?' en columnas categóricas? {(dt[categorical_cols] == '?').sum().sum()}")
print(f"¿Quedan NaN en todo dt? {dt.isnull().sum().sum()}")
print("\nTipos de datos finales:")
print(dt.dtypes.value_counts())
print("\nPrimeras 5 filas:")
print(dt.head())

Columnas categóricas a procesar: 9
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'income']
  [workclass] → 1836 faltantes, imputando con KNN-moda...
  [occupation] → 1843 faltantes, imputando con KNN-moda...
  [native_country] → 583 faltantes, imputando con KNN-moda...

✓ Imputación y codificación completada.
¿Quedan '?' en columnas categóricas? 0
¿Quedan NaN en todo dt? 0

Tipos de datos finales:
int64    14
Name: count, dtype: int64

Primeras 5 filas:
   age  workclass  education  education_num  marital_status  occupation  \
0   39          6          9             13               4           0   
1   50          5          9             13               2           3   
2   38          3         11              9               0           5   
3   53          3          1              7               2           5   
4   28          3          9             13               2           9   

   relationship  race  sex  c